**ADDESTRAMENTO ENCODER**

SETUP

In [ ]:
import digitalhub as dh
import pandas as pd
import matplotlib.pyplot as plt

NOME_PROGETTO = "floods"
project = dh.get_project(NOME_PROGETTO)
print(f"Progetto: {project.name}")

TRAINING

In [ ]:
# setup ambiente
encoders_train_func = project.new_function(
    name="encoders_train-job",
    kind="python",
    python_version="PYTHON3_10",
    code_src="3_training", 
    handler="cae.pretrain_encoders",
    requirements=["torch", "torchvision", "pandas", "numpy", "rasterio", "tqdm"] 
)

# parametri training
parametri = {
    "epochs": 200, "batch_size": 16, "lr": 1e-4, "weight_decay": 1e-4,      
    "patch_size": 256, 
    "n_images1": 4, "n_channels1": 2,                    # sar
    "n_images2": 4, "n_channels2": 10,                   # ottiche               
    "mamba": False, 
    "workers": 4
}

print(f"PARAMETRI: {parametri}")

run_train_encoders = encoders_train_func.run("job", parameters=parametri, wait=True)
print(f"STATO FINALE: {run_train_encoders.status.state}")

RISULTATI

In [ ]:
# salvataggio log
print("SALVATAGGIO METRICHE")
path_s1 = project.get_artifact("metrics-s1").download()
path_s2 = project.get_artifact("metrics-s2").download()

# risultati grafici
df_s1 = pd.read_csv(path_s1)
df_s2 = pd.read_csv(path_s2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# sar
ax1.plot(df_s1['epoch'], df_s1['train_loss'], color='blue', label='Train Loss')
ax1.set_title('Pre-training SAR')
ax1.set_xlabel('Epoche')
ax1.set_ylabel('Loss (MSE)')
ax1.grid(True)

# ottico
ax2.plot(df_s2['epoch'], df_s2['train_loss'], color='red', label='Train Loss')
ax2.set_title('Pre-training OTTICO')
ax2.set_xlabel('Epoche')
ax2.grid(True)

plt.show()